# Publication-safe notebook

This generalized reference implementation expects a live SparkSession named
`spark` and pseudonymous tables matching `docs/data-contract.md`. It contains no
institution-specific ingestion, credentials, internal paths, data, or results.

Parameter defaults are illustrative and intentionally differ from internal
operating settings. Calibrate them only with authorized development data.


In [ ]:
from pyspark.sql import functions as F

import os

DB = os.environ.get("FRAUD_GRAPH_WRITE_DB", "fraud_graph_demo")
EMP = os.environ.get("FRAUD_GRAPH_PREFIX", "onboarding_fraud")
EDGES = f"{DB}.{EMP}_v2_edges_transferred"
SUBG  = f"{DB}.{EMP}_v2_subgraph_node_index"

ANCHORS = {
    "202401": ["202310", "202311", "202312"],
    "202402": ["202311", "202312", "202401"],
    "202403": ["202312", "202401", "202402"],
}
CAPS = [int(value) for value in os.environ.get("FRAUD_GRAPH_SENSITIVITY_CAPS", "5,15,25,75").split(",")]

spark.conf.set("spark.sql.shuffle.partitions", os.environ.get("FRAUD_GRAPH_SHUFFLE_PARTITIONS", "200"))
print("cores cap:", spark.conf.get("spark.executor.cores"), "x max execs",
      spark.conf.get("spark.dynamicAllocation.maxExecutors"), "| caps:", CAPS)

def sweep_anchor(anchor, months):
    e0 = (spark.table(EDGES).where(F.col("txn_month").isin(*months))
          .where(F.col("src") != F.col("dst")).select("src", "dst").persist())
    lab   = spark.table(SUBG).where(F.col("anchor_month") == anchor)
    seeds = lab.where(F.col("is_known_bad") == 1).select(F.col("id").alias("src")).distinct()
    cands = (lab.where((F.col("is_candidate") == 1) & (F.col("is_known_bad") == 0))
             .select("id", F.col("target_fraud").cast("int").alias("y")).persist())
    fanin = e0.groupBy("dst").agg(F.countDistinct("src").alias("fi")).persist()

    reference_cap = CAPS[len(CAPS) // 2]
    tot = fanin.count()
    below = fanin.where(F.col("fi") <= reference_cap).count()
    nb  = cands.where(F.col("y") == 1).count(); ng = cands.where(F.col("y") == 0).count()
    print("\n=== anchor %s ===" % anchor)
    print("fan_in<=%d = the %.3fth percentile of beneficiaries | candidates: bad=%d good=%d"
          % (reference_cap, 100.0 * below / max(tot, 1), nb, ng))
    print("cap |  %bad_share   %good_share    ratio")
    for C in CAPS:
        rare       = fanin.where(F.col("fi") <= C).select("dst")
        seed_mules = e0.join(seeds, "src").select("dst").join(rare, "dst").distinct()
        sharers    = (e0.join(seed_mules, "dst").select(F.col("src").alias("id"))
                      .distinct().withColumn("sh", F.lit(1)))
        c2 = cands.join(sharers, "id", "left").na.fill({"sh": 0})
        bs = c2.where(F.col("y") == 1).agg(F.avg("sh")).first()[0] or 0.0
        gs = c2.where(F.col("y") == 0).agg(F.avg("sh")).first()[0] or 0.0
        mark = "  <- illustrative reference" if C == CAPS[len(CAPS) // 2] else ""
        print("%3d |   %6.2f%%      %6.3f%%    %5.1fx%s" % (C, 100 * bs, 100 * gs, (bs / gs if gs else 0), mark))
    e0.unpersist(); cands.unpersist(); fanin.unpersist()

for a, ms in ANCHORS.items():
    sweep_anchor(a, ms)

print("\nDONE. Compare stability across the configured caps; select no threshold without authorized validation.")
